# One LightGBM, from raw data: CV 0.94626 / LB 0.94633

Every notebook above 0.9464 on this leaderboard is a rank-average of other
notebooks. This one is a single model trained from `train.csv`, and it is built
so that it is *different* from the models those blends already contain — that is
the whole reason it exists.

I spent three days rank-averaging public submissions and measured, with twenty
submissions, that the approach tops out around 0.94644 no matter what you add
([details here](https://www.kaggle.com/code/megayak/s6e9-lb-0-94643-and-six-missing-sources)).
The sweeps also showed *why*: a new source only moves a blend if it is both
decorrelated from the blend (Spearman ≤ 0.998) **and** nearly as strong as the
blend on its own. Nothing public satisfied both. So I built one.

**Recipe.** Two feature families that do not overlap:

* the generator leak — low-order digits and moduli of income and commute, a
  quantisation ladder, exact-value frequency. On a plain pipeline this block was
  worth +0.0017 in my ablation. **On top of najiama's triple target encoding it is
  worth +0.000066 CV / +0.00003 LB** — measured by dropping it and resubmitting
  (0.94630 without, 0.94633 with). Most of what the digits know, a target encoder
  at exact-income resolution already knows. It stays because it is still positive
  on ten of ten folds, but the honest number is the small one.
* [@najiama](https://www.kaggle.com/code/najiama/pure-lgbm-model-cv-0-94606-lb-0-94637)'s
  V3 recipe — Markus.JM's "Smooth Keys" (income at exact / 100 / 1000 resolution
  as string categories), **triple target encoding** of every categorical key at
  smoothing `auto`, `10` and `100` simultaneously, original-dataset target means,
  and their LightGBM settings (`max_bin=1024`, `colsample_bytree=0.3`, depth 5)

Fold-safe throughout: `sklearn.preprocessing.TargetEncoder` refits inside every
outer fold with its own inner CV, so no validation label reaches an encoder.
Predictions are pooled in rank space per fold.

**OOF and test predictions are published** as a dataset, aligned to the frozen
`StratifiedKFold(10, shuffle=True, random_state=42)` partition over `train.csv`
in original row order, so this stacks row-for-row with anything else on that
partition. Blend it, stack it, check it.

In [ ]:
import glob, time, json
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import TargetEncoder
from scipy.stats import rankdata
import lightgbm as lgb

T0 = time.time()
log = lambda m: print(f"[{time.time()-T0:6.0f}s] {m}", flush=True)

def find(name):
    h = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    if not h:
        raise FileNotFoundError(name)
    return h[0]

TARGET = "Will_Buy_EV"
CATS = ["Gender", "City_Type", "Current_Car_Type", "Home_Charging_Possible",
        "Subsidy_Available", "Range_Anxiety_Level"]
NUMS = ["Age", "Annual_Income_USD", "Daily_Commute_km", "Number_of_Cars_Owned",
        "Charging_Stations_Near_Home", "Charging_Stations_Near_Work",
        "Environmental_Concern_Level"]
N_SPLITS, SEED = 10, 42
rk = lambda v: rankdata(v) / len(v)

train = pd.read_csv(find("train.csv")).drop(columns=["id"])
test  = pd.read_csv(find("test.csv")); test_id = test.pop("id").to_numpy()
orig  = pd.read_csv(find("EV_Adoption_and_Range_Anxiety_Dataset.csv")).drop(columns=["Buyer_ID"])
y = (train.pop(TARGET) == "Yes").astype(int).to_numpy()
log(f"train {train.shape} | test {test.shape} | original {orig.shape}")

## Features

In [ ]:
def build(train, test, orig):
    n = len(train)
    df = pd.concat([train, test], ignore_index=True)
    inc = df.Annual_Income_USD.to_numpy(np.int64)
    km  = np.round(df.Daily_Commute_km.to_numpy(float) * 10).astype(np.int64)

    # --- the generator leak: low-order structure of the continuous columns ---
    df["inc_d1"] = (inc % 10).astype("int8")
    df["inc_d2"] = (inc // 10 % 10).astype("int8")
    df["inc_d3"] = (inc // 100 % 10).astype("int8")
    df["inc_mod100"]  = (inc % 100).astype("int16")
    df["inc_mod1000"] = (inc % 1000).astype("int16")
    df["km_d1"]     = (km % 10).astype("int8")
    df["km_mod100"] = (km % 100).astype("int8")
    for d in (50, 100, 250, 500, 1000, 2500, 5000):
        df[f"inc_q{d}"] = (inc // d).astype("int32")
    for d in (5, 10, 25, 50):
        df[f"km_q{d}"] = (km // d).astype("int32")
    s = pd.Series(inc); df["fq_inc"] = s.map(s.value_counts()).astype("float32").values
    s = pd.Series(km);  df["fq_km"]  = s.map(s.value_counts()).astype("float32").values

    # --- najiama's flags ---
    df["is_30k_spike"]        = (inc == 30000).astype("int8")
    df["is_millionaire_cliff"] = (inc >= 170537).astype("int8")
    df["is_dead_zone"]        = ((inc >= 38000) & (inc <= 42000)).astype("int8")
    df["is_env_hater"]        = (df.Environmental_Concern_Level == 1).astype("int8")

    # --- original-dataset target means (najiama) ---
    og = orig.dropna(subset=["Annual_Income_USD", "Daily_Commute_km",
                             "Environmental_Concern_Level"]).copy()
    og[TARGET] = (og[TARGET] == "Yes").astype(int)
    gm = og[TARGET].mean()
    for c in CATS + NUMS:
        df[f"{c}_org_mean"] = df[c].map(og.groupby(c)[TARGET].mean()).fillna(gm).astype("float32")

    # --- keys for triple target encoding: Smooth Keys + cats + low-card numerics ---
    K = pd.DataFrame({
        "k_inc_exact": inc.astype(str),
        "k_inc100":    (inc // 100).astype(str),
        "k_inc1000":   (inc // 1000).astype(str),
        "k_km_int":    (km // 10).astype(str),
    })
    for c in CATS + ["Age", "Number_of_Cars_Owned", "Charging_Stations_Near_Home",
                     "Charging_Stations_Near_Work", "Environmental_Concern_Level"]:
        K[f"k_{c}"] = df[c].astype(str).to_numpy()
    for c in K.columns:                      # label-free frequency of each key
        df[f"{c}_fe"] = K[c].map(K[c].value_counts(normalize=True)).astype("float32").values
    for c in CATS:
        df[c] = df[c].astype("category")

    cut = lambda x: (x.iloc[:n].reset_index(drop=True), x.iloc[n:].reset_index(drop=True))
    (X, Xte), (Kt, Kte) = cut(df), cut(K)
    return X, Xte, Kt, Kte

X, Xte, K, Kte = build(train, test, orig)
log(f"features {X.shape[1]} | target-encoded keys {K.shape[1]} x 3 smoothings")

## Ten folds, triple target encoding refit per fold

In [ ]:
PARAMS = dict(n_estimators=20000, learning_rate=0.02, max_depth=5, num_leaves=32,
              min_child_samples=10, subsample=0.8, subsample_freq=1,
              colsample_bytree=0.3, reg_alpha=0.071, reg_lambda=2.0, max_bin=1024,
              feature_pre_filter=False, n_jobs=-1, verbose=-1)

cv  = StratifiedKFold(N_SPLITS, shuffle=True, random_state=42)
oof = np.zeros(len(X)); pte = np.zeros(len(Xte)); iters = []

for f, (a, b) in enumerate(cv.split(X, y)):
    A, B, C = X.iloc[a].copy(), X.iloc[b].copy(), Xte.copy()
    for smooth, tag in (("auto", "auto"), (10.0, "10"), (100.0, "100")):
        te = TargetEncoder(shuffle=True, cv=5, smooth=smooth, random_state=42)
        ea = te.fit_transform(K.iloc[a], y[a])          # inner CV: fold-safe
        eb, ec = te.transform(K.iloc[b]), te.transform(Kte)
        for i, c in enumerate(K.columns):
            A[f"{c}_te{tag}"] = ea[:, i].astype("float32")
            B[f"{c}_te{tag}"] = eb[:, i].astype("float32")
            C[f"{c}_te{tag}"] = ec[:, i].astype("float32")

    m = lgb.LGBMClassifier(random_state=SEED, **PARAMS)
    m.fit(A, y[a], eval_set=[(B, y[b])], eval_metric="auc",
          callbacks=[lgb.early_stopping(500, verbose=False)])
    pb = m.predict_proba(B)[:, 1]
    oof[b] = rk(pb)                                     # rank inside the fold
    pte   += rk(m.predict_proba(C)[:, 1]) / N_SPLITS
    iters.append(m.best_iteration_)
    log(f"fold {f}  auc {roc_auc_score(y[b], pb):.5f}  trees {m.best_iteration_}")

auc = roc_auc_score(y, oof)
print(f"\nOOF AUC = {auc:.6f}")

In [ ]:
pd.DataFrame({"id": test_id, TARGET: pte}).to_csv("submission.csv", index=False)
np.save("oof_hybrid.npy", oof)
np.save("test_hybrid.npy", pte)
json.dump({"oof_auc": float(auc), "iters": iters, "n_splits": N_SPLITS, "seed": SEED},
          open("report.json", "w"), indent=2)
log("wrote submission.csv, oof_hybrid.npy, test_hybrid.npy")

## What this is for

Two things, in order.

**As a model**: CV 0.94626 from one LightGBM with no blending. Fork it and it
runs in about ten minutes on CPU.

**As a blend member**: this is the point. The public blends are saturated because
every candidate that passes the correlation gate is too weak, and every strong
candidate is too correlated. This model was built from a feature family the
blends do not contain, on top of the recipe that produced their strongest member.
`oof_hybrid.npy` / `test_hybrid.npy` are in the output and in the companion
dataset — they line up positionally with
[@dariushafshar's OOF library](https://www.kaggle.com/datasets/dariushafshar/s6e9-golem-oof-library)
and [@najiama's OOF files](https://www.kaggle.com/datasets/najiama/s6e9-oof)
on the same frozen partition.

One caveat, disclosed the way the golem library discloses its own: early stopping
here watches the held-out fold, so the OOF is mildly optimistic. If you stack on
it, weight it accordingly.

---

Credit: the triple-target-encoding recipe, the Smooth Keys, the flags and the
LightGBM settings are [@najiama](https://www.kaggle.com/najiama)'s V3, which
itself builds on [@maiernator](https://www.kaggle.com/maiernator)'s Smooth Keys;
the source dataset is [@itzzomkar](https://www.kaggle.com/datasets/itzzomkar/ev-adoption-behavior-and-range-anxiety)'s.
The digit-leak features and the measurement that motivated combining them are mine.
If this was useful, an upvote helps.